In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API KEY SET")

API KEY SET


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

llm.invoke("Hello!").content

'Hello. How can I assist you today?'

## **RAG IMPLEMENTATION WITH OUR OWN DATA** 

## STEP 1 - Preparing Document for our text

In [6]:
from langchain_core.documents import Document

my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human
"""

docs = [Document(page_content=my_text, metadata={"source" : "ABC", "documentID" : "Doc1"})]

docs

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language process

## STEP 2 - Splitting the Document into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'ABC', 'documentID': '

## STEP 3 - Creating Embeddings for the chunks

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

f:\AI_Projects\rag_basics\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\harsh\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5579.40it/s]


In [2]:
embedding_model.embed_query("What is AI??")

[-0.022800249978899956,
 -0.014960713684558868,
 -0.003175379941239953,
 0.01735958270728588,
 0.013784509152173996,
 0.0017524594441056252,
 0.094301238656044,
 0.030132684856653214,
 0.03359249606728554,
 0.05015338212251663,
 -0.020683130249381065,
 -0.018973801285028458,
 0.02717948891222477,
 -0.050934769213199615,
 -0.07851000875234604,
 0.03863966464996338,
 -0.01049608364701271,
 -0.05362080782651901,
 -0.0996120497584343,
 -0.05779704451560974,
 0.00031567097175866365,
 0.021490376442670822,
 -0.050385091453790665,
 -0.04242037981748581,
 -0.030625803396105766,
 0.09469327330589294,
 0.0028078865725547075,
 -0.06436789780855179,
 0.002770352177321911,
 -0.013305850327014923,
 0.006962461397051811,
 0.016883036121726036,
 0.09239822626113892,
 0.026143422350287437,
 -0.06658580154180527,
 0.02727467380464077,
 -0.03465939313173294,
 -0.024630526080727577,
 0.058106642216444016,
 -0.036757003515958786,
 -0.021397924050688744,
 -0.045401982963085175,
 0.020491650328040123,
 -0.07

## STEP 4 - Create & Store Embeddings in Vector Store

In [8]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

## STEP 5 - Semantic Search

In [9]:
vectorstore.similarity_search("What is AI?", k=3)

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human'),
 Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='The traditional goals of A

## TALK TO LLM

In [10]:
context = vectorstore.similarity_search("What is AI?", k=3)

response = llm.invoke(f"What is AI? You can answer using the following context: {context}")

print(response.content)

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence. This includes:

1. Learning: The ability of machines to acquire new knowledge and skills.
2. Reasoning: The ability of machines to draw conclusions and make decisions based on available information.
3. Problem-solving: The ability of machines to identify and solve complex problems.
4. Perception: The ability of machines to interpret and understand their environment.
5. Decision-making: The ability of machines to make choices that maximize their chances of achieving defined goals.

AI is a field of research that combines engineering, mathematics, and computer science to develop and study methods and software that enable machines to use learning and intelligence to take actions. The goals of AI research include:

1. Learning
2. Reasoning
3. Knowledge representation
4. Planning
5. Natural language processing
6. Perception
7. Support for robotics

To achiev